# QUESTION 2(b) — Credit Card Fraud Detection

### Task
Using `credit.csv`, build a reproducible Decision Tree workflow with stratified train/test splitting, GridSearchCV, and two appropriate metrics for an imbalanced fraud problem.

## Exam-ready algorithm

```text
START

1. Load credit.csv.

2. Separate the data:
      X = predictor variables
      y = binary fraud target

3. Split the data using train_test_split:
      X_train, X_test, y_train, y_test
      Use stratify = y
      Use a fixed random_state, e.g. 42

4. Create a baseline Decision Tree Classifier:
      DecisionTreeClassifier(random_state=42)

5. Define a hyperparameter grid containing at least:
      max_depth
      min_samples_leaf

6. Create GridSearchCV using the Decision Tree and
   the hyperparameter grid.

7. Fit GridSearchCV using X_train and y_train ONLY.
   Use cross-validation, for example cv=5.

8. Select the best model using best_estimator_.

9. Predict the held-out test set:
      y_pred = best_model.predict(X_test)

10. Calculate exactly two appropriate metrics:
       Precision
       Recall

11. Report the two test-set results.

END
```

In [ ]:
# Optional Python template
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score, recall_score

df = pd.read_csv("credit.csv")
X = df.drop(columns=["Fraud"])
y = df["Fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

tree = DecisionTreeClassifier(random_state=42)
    
param_grid = {
    "max_depth": [3, 5, 10, None],
    "min_samples_leaf": [1, 5, 10, 20]
}

grid = GridSearchCV(tree, param_grid, cv=5, scoring="f1")
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))

# QUESTION 5(c) — Home Credit Preprocessing Pipeline

### Task
Using `home.csv`, automatically identify numerical and categorical features, preprocess each type appropriately, combine the transformations with `ColumnTransformer`, place a classifier inside a parent `Pipeline`, and evaluate on held-out test data.

## Exam-ready algorithm

```text
START

1. Load home.csv.

2. Separate the data:
      X = predictor variables
      y = binary target variable

3. Automatically identify feature types:
      numerical_features = numerical columns
      categorical_features = categorical/object columns

4. Create numerical preprocessing:
      a. Impute missing numerical values using the median.
      b. Standardize numerical variables using StandardScaler.

5. Create categorical preprocessing:
      a. Impute missing categorical values using the most frequent value.
      b. One-hot encode categorical variables.
      c. Ignore unseen categories using handle_unknown='ignore'.

6. Combine the two preprocessing procedures using ColumnTransformer:
      Numerical columns → numerical pipeline
      Categorical columns → categorical pipeline

7. Create a classifier, for example:
      DecisionTreeClassifier(random_state=42)

8. Create one parent Pipeline containing:
      preprocessing → classifier

9. Split the data into training and held-out test sets.
      Use stratify=y if appropriate for the binary target.
      Use random_state=42.

10. Fit the complete pipeline on X_train and y_train.

11. Predict the held-out test data using the fitted pipeline.

12. Evaluate using appropriate classification metrics,
    such as precision, recall and F1-score.

END
``

In [ ]:
# Optional Python template
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score, recall_score, f1_score

df = pd.read_csv("home.csv")
    
X = df.drop(columns=["TARGET"])
y = df["TARGET"]

numeric_features = X.select_dtypes(include="number").columns
categorical_features = X.select_dtypes(exclude="number").columns

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(random_state=42))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))

# QUESTION 6(d) — Custom Probability Threshold

### Task
Obtain predicted probabilities using `.predict_proba()`, apply a custom threshold, calculate two metrics, and explain how to evaluate many thresholds systematically.

## Exam-ready algorithm

```text
START

1. Use the trained classifier to obtain probabilities:
      probabilities = model.predict_proba(X_test)[:, 1]
   The [:, 1] selects the probability of the positive class,
   such as fraud.

2. Choose a custom threshold, for example:
      threshold = 0.80

3. Convert probabilities into binary predictions:
      If probability >= threshold → predict 1
      Otherwise → predict 0

4. Calculate two appropriate metrics:
      Precision
      Recall

5. Create an array of thresholds, for example:
      [0.10, 0.20, 0.30, ..., 0.90]

6. Loop through every threshold.

7. For each threshold:
      a. Convert probabilities to binary predictions.
      b. Calculate precision.
      c. Calculate recall.
      d. Store the results.

8. Compare the results across thresholds.

9. Select an operational threshold based on:
      - desired precision/recall balance;
      - false-positive cost;
      - false-negative cost;
      - business requirements.

END
```


In [ ]:
# Optional Python template
from sklearn.metrics import precision_score, recall_score

y_prob = model.predict_proba(X_test)[:, 1]

threshold = 0.80
y_pred = (y_prob >= threshold).astype(int)

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

print("Precision:", precision)
print("Recall:", recall)

# Evaluate a range of thresholds
thresholds = [0.10, 0.20, 0.30, 0.40, 0.50,
              0.60, 0.70, 0.80, 0.90]

for threshold in thresholds:
    y_pred = (y_prob >= threshold).astype(int)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    print(threshold, precision, recall)

# QUESTION 7(c) — House Prices Regression

### Task
Using `prices.csv`, partition the data, scale features appropriately, fit at least two regression models, generate test predictions, and report MAE, RMSE and R².

## Exam-ready algorithm

```text
START

1. Load prices.csv.

2. Separate the data:
      X = predictor variables
      y = SalePrice target

3. Split the dataset:
      X_train, X_test, y_train, y_test
      Use train_test_split
      Use random_state=42

4. Identify numerical and categorical features.

5. Preprocess the features:
      - Impute missing values where necessary.
      - Encode categorical variables.
      - Scale numerical features where appropriate.

6. Create regression models, for example:
      OLS / LinearRegression
      Ridge
      Lasso

7. Place preprocessing and each regression model
   inside a Pipeline.

8. Fit the selected regression models using training data.

9. Generate predictions on X_test.

10. Calculate for each model:
       MAE
       RMSE
       R²

11. Report the exact metric values for each model.

END
```



In [ ]:
# Optional Python template for the regression question
import numpy as np
import pandas as pd
    
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv("prices.csv")
X = df.drop(columns=["SalePrice"])
y = df["SalePrice"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

numeric_features = X.select_dtypes(include="number").columns
categorical_features = X.select_dtypes(exclude="number").columns

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

models = {
    "OLS": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.01, max_iter=10000)
}

for name, estimator in models.items():
    model = Pipeline([
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    print(name)
    print("MAE:", mae)
    print("RMSE:", rmse)
    print("R2:", r2)


# FINAL EXAM MEMORY SHEET

## 1. Fraud + GridSearchCV
**Load → X/y → stratified split → Decision Tree → max_depth/min_samples_leaf → GridSearchCV on training → best model → test → Precision + Recall**

## 2. Preprocessing Pipeline
**Identify columns → numerical: median + scale → categorical: most-frequent + one-hot → ColumnTransformer → classifier → Pipeline → train → test → metrics**

## 3. Probability Threshold
**predict_proba → select positive-class probability → threshold → binary predictions → Precision + Recall → loop over thresholds → compare → choose using costs/business requirements**

## 4. Regression
**Load → X/y → train/test split → preprocessing/scaling → OLS/Ridge/Lasso → fit → predict → MAE/RMSE/R²**

### Three rules to never forget
1. **Fit/tune using training data; keep the test set for final evaluation.**
2. **Use pipelines so preprocessing is applied consistently and can be safely used with cross-validation.**
3. **For imbalanced fraud classification, accuracy alone can be misleading; precision and recall are important.**
